### **Retrieval**

#### **A typical RAG pipeline**

**RAG Pipeline**

```text
Documents
   ↓
Loading
   ↓
Chunking
   ↓
Embedding
   ↓
Vector Store
   ↓
Retriever
   ↓
LLM
   ↓
Final Answer

#### **1) What is a retriever?**

##### **A) Definition**

A retriever takes a user query and returns the most relevant documents or chunks from a knowledge source(VectorDB or VectorStore)

A retriever normally does not generate the final answer itself; it collects the relevant context required to generate the answer.

A retriever is a component that:
- Takes a user query 

- Searches the knowledge source 

- Returns the most relevant documents

A retriever’s job is to find relevant information based on the user’s question.

- User question
- Retriever searches the stored documents
- Returns the most relevant chunks 
- The LLM uses those chunks to generate the answer

**Simple example**

Suppose a vector database contains 1,000 chunks extracted from PDF documents.

The user asks: "What is semantic chunking?"

The retriever does not send all 1,000 chunks to the LLM. It searches the database and returns only the relevant chunks (context/retrieved data/ranked data):

- Chunk 12 → Definition of semantic chunking
- Chunk 48 → Example of semantic chunking
- Chunk 91 → Advantages of semantic chunking

These relevant chunks are then passed to the LLM.

**Simple analogy**

Think of a retriever as a librarian:
- User = Student
- Documents = Library books
- Retriever = Librarian
- LLM = Teacher

The student asks a question. 

The librarian finds the most relevant books or pages and gives them to the teacher. 

The teacher reads those pages and generates the final answer.

**Dummy Indicative code**

retriever = vector_store.as_retriever(
 search_type="mmr",
 search_kwargs={
 "k": 4, # Final number of documents to return
 "fetch_k": 20 # Candidate documents considered by MMR
 }
)

*Retrieve relevant documents*

documents = retriever.invoke("What is a vector database?")

*Display the retrieved content and metadata => Relevant context for the LLM to answer*

for document in documents:
 print(document.page_content)
 print(document.metadata)
 print("-" * 50)


**Parameters in a dummy code**
- k = final number of results retrieved from vectorstore
- filter = metadata restriction
- score_threshold = minimum acceptable relevance score. Referes on the lines of similarity search say for example cosine similarity between -1 to 1(full match) 
- fetch_k = number of candidates fetched before MMR. 
- lambda_mult = balance between relevance and diversity in MMR(maximum marginal retrieval). MMR is advanced type of similarity search which works on cosine similarity itself but in a diff way. 
- search_type → decides which search algorithm runs, out of three available i.e. cosine similarity, dot product, Euclidian distance. 

xyz
- similarity → returns the most similar chunks
- similarity_score_threshold → returns only the chunks that pass the threshold
- mmr → returns relevant and diverse chunks
- search_kwargs → configures the selected search algorithm

##### **B) Similarity Search Methods**

The most commonly supported similarity metrics :

**1) Cosine Similarity** 

Measures the angle/direction similarity between two vectors.
- Higher score = More similar 
- Most commonly used for text embeddings

**2) Euclidean Distance — L2**

Measures the straight-line distance between two vectors.
- Smaller distance = More similar

**3) Dot Product / Inner Product**

Multiplies corresponding vector values and adds them.

- Higher score = More similar
- With normalized vectors, dot product becomes equivalent to cosine similarity.


**Summary**

| Method | Best result |
|---|---|
| Cosine Simialirty | `Highest Score` |
| Euclidean distance | `Lowest Distance` |
| Dot Product | `Highest Score` |

##### **C) Metadata Filtering**

A retriever should not rely only on semantic similarity. It should also support structured constraints using document metadata.
Metadata helps the retriever limit the search to documents that satisfy specific conditions such as department, year, document type, year, source, user role, etc.


**Example Metadata**

metadata = {
 "department": "HR",
 "year": 2026,
 "document_type": "policy",
 "access_role": "manager"
}

*User Query*: Show the HR leave policy for 2026.

Metadata Filter

filter = {
 "department": "HR",
 "year": 2026
}

The retriever will search only the documents whose metadata matches:
department = HR
year = 2026

It will ignore documents from other departments or years, even when their content is semantically similar to the query.

**Common Types of Metadata Filters**
1) *Exact-match filters* : 
Match an exact metadata value.
{"department": "HR"}

2) *Range filters* : 
Match values within a range.
{"year": {"$gte": 2024, "$lte": 2026}}

3) *Boolean filters* : 
Combine multiple conditions using AND, OR, or NOT.
{
 "$and": [
 {"department": "HR"},
 {"year": 2026}
 ]
}
4) *Date filters* : Retrieve documents created or updated within a specific date range.

5) *Department filters* : 
Restrict retrieval to departments such as HR, Finance, Legal, or Engineering.
6) *Document-type filters* : 
Restrict retrieval to policies, reports, invoices, manuals, or contracts.
7) *Source filters* :
Search only selected PDFs, websites, databases, or repositories.
8) *Tenant filters* :
Ensure that users can retrieve documents only from their own organization or tenant.
9) *Role-based access filters* :
Restrict documents according to roles such as employee, manager, HR, or administrator.
10) *Pre-filtering and post-filtering* :
Decide whether metadata restrictions are applied before or after the retrieval operation. 

Metadata Filtering broadly can be categorised into 2 types : 

A) Pre-filtering : Filter first → Search later

B) Post-filtering : Search first → Filter later

**Pre-filtering**

It means applying metadata conditions before running vector or keyword search. 

All stored documents ==> Apply metadata filter ==> Allowed documents only ==> Similarity or keyword search ==> Final results

Example: 

Suppose the vector database contains 10,000 documents:
- HR documents = 1,000
- Finance documents = 3,000
- Engineering documents = 4,000
- Legal documents = 2,000

The user asks:

Show the HR leave policy for 2026.

The filter is:
filter = {
 "department": "HR",
 "year": 2026
}

With pre-filtering:
```
10,000 documents
 ↓
Filter department = HR and year = 2026
 ↓
Only 150 permitted documents remain
 ↓
Similarity search runs on those 150 documents
 ↓
Most relevant HR leave-policy chunks are returned
```


**Code Example**

retriever = vector_store.as_retriever(
 search_type="similarity",
 search_kwargs={
                "k": 4,
                "filter": {
                            "department": "HR",
                            "year": 2026
                            }
                }
                                    )

documents = retriever.invoke("Show the HR leave policy for 2026.")

**Advantages of Pre-filtering**
- Searches a smaller document set
- Reduces irrelevant results
- Improves security
- Supports tenant isolation
- Prevents unauthorized documents from entering the candidate list
- Can improve retrieval speed 

**Post-filtering**

Post-filtering means running retrieval first and applying metadata conditions afterward.

```
All stored documents
 ↓
Similarity or keyword search
 ↓
Top candidate documents
 ↓
Apply metadata filter
 ↓
Final allowed results
```

**Example**

Suppose the retriever first returns the top five semantically similar documents:

Result 1 → Finance leave policy, 2026

Result 2 → HR leave policy, 2025

Result 3 → Legal leave guideline, 2026

Result 4 → HR leave policy, 2026

Result 5 → Engineering leave policy, 2026

Now the filter is applied:

filter = {
 "department": "HR",
 "year": 2026
}

After post-filtering, only one result remains:

Result 4 → HR leave policy, 2026

The retriever originally fetched five documents, but four were removed after retrieval.

**Code Example**
documents = vector_store.similarity_search(
 "Show the HR leave policy for 2026.",
 k=5
)

filtered_documents = [
 document
 for document in documents
 if document.metadata.get("department") == "HR"
 and document.metadata.get("year") == 2026
]

**Limitations of Post-filtering**

- It may return too few final results
- Relevant permitted documents may never enter the initial top-k
- Unauthorized documents may enter the intermediate candidate set
- It is less suitable for strict access control
- A larger initial k may be required

For example:
```
Initial retrieval returns top 5
 ↓
4 results fail the filter
 ↓
Only 1 final result remains
```
Even though more valid HR documents may exist in the database, they may not have appeared in the original top five

### **2) Retrieval Types**

#### **2A) Sparse Retrieval**

Sparse retrieval searches using exact keywords and term matching. 

It is an old but reliable technique for retrieving the information. 

It doesn't use dense embedding but uses sparse vectors. 

This method is used to create Vectorless RAG. 

**Example**

Query: "employee leave policy"

Returns documents containing words such as: employee, leave, policy

Common methods:

- BM25
- TF-IDF
- Keyword Search

#### **2B) Dense Retrieval**

Dense retrieval uses embeddings to understand the semantic meaning of the query.

**Example**

Query: "How many days off can employees take?"

It can retrieve:"Employees are entitled to 20 days of annual leave."

The exact words may be different, but the meaning is similar

#### **2C) Hybrid Retrieval**

Hybrid retrieval combines sparse and dense retrieval.

Keyword/BM25 Search (Sparse)

    +

Vector Search (Dense)

    ↓

`Combined Results`

**Example**:

Query: "HR leave policy 2026"

Sparse retrieval matches exact terms such as:
HR
leave policy
2026

Dense retrieval finds semantically similar content such as:
employee annual vacation guidelines

Hybrid retrieval combines both results for better accuracy.

### **3) Query Transformation**

Query transformation improves the user’s original query before retrieval so the system can find more relevant information. It may rewrite the query, add related
terms, or break a complex query into smaller questions. 

#### **3A) Query Rewriting**

Query rewriting converts an unclear, incomplete, or conversational query into a clearer standalone search query.

**Example** :

*Original query*:

"What did he say about it?"

*Rewritten query*:

"What did the CEO say about the 2026 acquisition?"
This is especially useful in conversational RAG, where the current question depends on previous messages.

#### **3B) Query Expansion**

Query expansion adds synonyms, related terms, acronyms, spelling variations, or alternative phrases to the original query.

**Example**:

*Original query*:

"employee leave policy"

*Expanded query*:

"employee leave policy OR vacation policy OR annual leave guidelines"

**Another example:**

*Original term*:

"car"

*Expanded terms*:

"car, automobile, vehicle"

Query expansion broadens the search and helps retrieve documents that use different words for the same concept.

```
Original query
 ↓
Add related terms or synonyms
 ↓
Broader retrieval
```

#### **3C) Query Decomposition**

Query decomposition breaks a complex question into smaller and simpler sub-questions.

**Example**:

*Original query*:

"Compare the revenue of Company A and Company B in 2025 and explain why their growth rates were different."

It can be decomposed into:

Sub-query 1:
What was Company A's revenue in 2025?

Sub-query 2:
What was Company B's revenue in 2025?

Sub-query 3:
What factors affected Company A's growth?

Sub-query 4:
What factors affected Company B's growth?

The system retrieves information for each sub-query and combines the results to answer the original question. Query decomposition is useful for comparison, multi - hop, and complex questions. 

```
Complex query
 ↓
Multiple smaller sub-queries
 ↓
Retrieve evidence for each query
 ↓
Combine the results
```

### **4) Re-ranking**

Reranking is a second-stage retrieval process that takes an initial set of retrieved documents, calculates a more accurate relevance score for each document basis teh user query, and reorders arranges them from most relevant to least relevant and the most relevants ones are sent to the LLM. 

```
User Query
 ↓
Initial Retriever
(BM25, Vector Search, or Hybrid Search)
 ↓
Top Candidate Documents
 ↓
Reranker
 ↓
Reordered by Relevance
 ↓
Top Documents Sent to the LLM
```


##### **4A) Why do we need Re-ranking**

The initial retriever must search thousands or millions of documents quickly. Therefore, it normally uses a fast retrieval method such as:
- BM25
- Vector similarity search
- Hybrid retrieval

Fast retrieval provides good candidates, but their original order may not be perfectly accurate.

A reranker applies a more powerful model only to this smaller candidate set. 

This creates a practical balance:
- Initial Retrieval → Fast and broad
- Reranking → Slower but more accurate

Production search systems use this multi-stage architecture because an expensive ranking model can be applied to a small candidate set rather than the entire
document collection. 

##### **4B) Simple Example of Re-ranking**

Suppose the user asks:

How many days of annual leave do employees receive?

The initial retriever returns these candidates:
1. Remote Work Policy
2. Sick Leave Policy
3. Annual Leave Policy
4. Leave Carry-Forward Policy
5. Employee Attendance Policy

These documents are related to employees and leave, but the most useful document is not ranked first.

The reranker evaluates every candidate against the original query:

- Annual Leave Policy → 0.95
- Leave Carry-Forward Policy → 0.76
- Sick Leave Policy → 0.31
- Employee Attendance Policy → 0.19
- Remote Work Policy → 0.08

The reranker then produces a better order:
1. Annual Leave Policy
2. Leave Carry-Forward Policy
3. Sick Leave Policy
4. Employee Attendance Policy
5. Remote Work Policy

Only the highest-ranked documents are passed to the LLM. 

##### **4C) How does a Cross-encoder Re-ranker works?**

This is based on the cross-attention technique. 

A standard embedding retriever usually encodes the query and documents separately
```
Query → Query Vector
 +
Document → Document Vector
 ↓
 Similarity Score
```

Because document vectors can be created and stored in advance, this approach is fast enough to search large collections.

A cross-encoder reranker processes the query and one candidate document **together**:

```
[Query + Candidate Document]
 ↓
 Cross-Encoder
 ↓
 Relevance Score
```

This joint processing allows the model to examine detailed relationships between the words in the query and the document. 
However, every query-document pair must be processed separately, making cross-encoders more computationally expensive than first-stage vector retrieval.

For 20 candidate documents, the reranker conceptually evaluates:
```
Query + Document 1 → Score
Query + Document 2 → Score
Query + Document 3 → Score
...
Query + Document 20 → Score
```
It then sorts the documents by these scores


##### **4D) Re-ranking based Real Production Flow**

```
1. User submits a query
            ↓
2. Apply metadata and security filters
            ↓
3. Retrieve a broad candidate set (using BM25, vector, or hybrid search)
            ↓
4. Send the candidate set to a reranker
            ↓
5. Calculate query-document relevance scores
            ↓
6. Sort candidates by reranker score
            ↓
7. Apply an optional minimum-score threshold
            ↓
8. Send the best documents to the LLM
```

**For example**:
```
1,000,000 stored chunks
 ↓
Retriever selects 50 candidates
 ↓
Reranker reorders those 50 candidates
 ↓
Top 5 chunks are passed to the LLM
```


The values 50 and 5 are only examples. In production, candidate count and final result count are tuned according to accuracy, latency, model limits, token budget, and
cost. 

Elasticsearch exposes this candidate window as rank_window_size, and reranking services accept a query plus a candidate document list and return relevance -
ranked results. 

##### **4E) Was Re-ranking introduced for RAG**

No. Reranking existed in information retrieval and search systems before modern RAG. Traditional search systems already used multi-stage or cascade ranking:

```
Fast candidate generation
 ↓
More accurate ranking stages
 ↓
Final search results
```
- The 2011 work A Cascade Ranking Model for Efficient Ranked Retrieval formalized a multi-stage ranking architecture designed to balance search effectiveness and
computational efficiency.
- In 2019, Passage Re-ranking with BERT demonstrated that BERT could be adapted to score query-passage pairs and substantially improve passage-ranking
performance. This work helped popularize transformer-based semantic reranking, but it did not invent the general reranking concept.

RAG later adopted the same established idea:

##### **4F) Benefits of Re-ranking**

Reranking can:
- Improve the ordering of retrieved documents
- Remove weak candidates using a relevance threshold
- Reduce irrelevant context sent to the LLM
- Reduce unnecessary input tokens
- Improve evidence quality
- Work on results from sparse, dense, or hybrid retrieval


Official production implementations such as Elasticsearch and Cohere accept an initial candidate set and reorder it according to query relevance; Elasticsearch also
supports candidate-window size, score thresholds, filters, and chunk-level rescoring for long documents. 

##### **4G) Limitations of Re-ranking**

Reranking adds:
- Additional latency
- Additional computation
- Additional inference cost

Therefore, it should normally be applied only to a limited candidate set, not to every document in the database.

### **5) Multimodal-Retriever**

A multimodal retriever retrieves relevant information across different modalities, such as text and images. 

It uses multimodal embedding models to represent compatible modalities in a shared vector space, allowing text-to-image, image-to-text and image-to-image similarity search. 

**Example**
Text-to-Image

```
Text Query
 ↓
Text Encoder
 ↓
Shared Embedding Space
 ↓
Search Image Vectors
 ↓
Relevant Images
Image-to-Image
Image Query
 ↓
Image Encoder
 ↓
Image Embedding Space
 ↓
Search Image Vectors
 ↓
Similar Images
```

### **6) Practical**

##### **6.1) Imports**

In [6]:
from pathlib import Path
import getpass
import os
import shutil

import numpy as np
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [7]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


##### **6.2) Reading & Loading the file**

In [8]:
path1=os.path.join(os.path.dirname(os.getcwd()),"Research")
path1

'c:\\Data_science\\Modern_route\\Research'

In [9]:
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()),"Research")

preferred_pdf = os.path.join(DATA_DIR, "LLAMA2_research_paper.pdf")

if os.path.exists(preferred_pdf):
    PDF_PATH = preferred_pdf
else:
    # Automatically find the PDF if its filename is slightly different
    available_pdfs = [f for f in os.listdir(DATA_DIR) if f.endswith(".pdf")]

    if len(available_pdfs) == 1:
        PDF_PATH = available_pdfs[0]
    elif len(available_pdfs) == 0:
        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )
    else:
        raise RuntimeError(
            "Multiple PDF files were found. Please set PDF_PATH manually.\n"
            + "\n".join(str(path) for path in available_pdfs)
        )

print("PDF found:")
print(PDF_PATH)

PDF found:
c:\Data_science\Modern_route\Research\LLAMA2_research_paper.pdf


In [10]:
loader = PyPDFLoader(str(PDF_PATH))

pages = loader.load()

print(f"Total PDF pages loaded: {len(pages)}")

Total PDF pages loaded: 77


In [11]:
print("First-page metadata:")
print(pages[0].metadata)

print("\nFirst 1,000 characters:")
print(pages[0].page_content[:1000])

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'c:\\Data_science\\Modern_route\\Research\\LLAMA2_research_paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann

##### **6.3) Metadata Enrichment**

In [12]:
def identify_section(paper_page: int) -> str:
    """
    Identify the major section of the Llama 2 paper
    using its printed PDF page number.
    """

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"

In [13]:
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = int(page_document.metadata.get("page", 0))
    paper_page = page_index + 1

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

In [14]:
for page_document in pages[:5]:
    print(page_document.metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'c:\\Data_science\\Modern_route\\Research\\LLAMA2_research_paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'c:\\Data_science\\Mo

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'c:\\Data_science\\Modern_route\\Research\\LLAMA2_research_paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}

##### **6.4) Chunking & Chunks metadata generation**

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Total pages: 77
Total chunks: 343


In [16]:
for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"llama2-page-{paper_page}-chunk-{chunk_number}"
    )

In [17]:
print("Chunk content:")
print(chunks[0].page_content[:1000])

print("\nChunk metadata:")
print(chunks[0].metadata)

Chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojni

##### **6.5) Embeddings generation**

In [18]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [19]:
test_vector = embeddings.embed_query("What is Llama 2?")

print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

Embedding dimensions: 1536
First 10 values: [0.0027942657470703125, -0.0521240234375, -0.021087646484375, -0.055419921875, -0.026397705078125, 0.028961181640625, -0.002071380615234375, 0.034759521484375, -0.0164794921875, -0.0245208740234375]


##### **6.6) VectorStore Creation**

In [20]:
PERSIST_DIRECTORY = os.path.join(os.getcwd(), "chroma_llama2_retriever")

# Set this to False when you want to reuse the existing index.
REBUILD_INDEX = True

if REBUILD_INDEX and os.path.exists(PERSIST_DIRECTORY):
    shutil.rmtree(
        PERSIST_DIRECTORY,
        ignore_errors=True
    )

In [21]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="llama2_retriever_demo",
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
)

print("Vector store created successfully.")
print(f"Stored chunks: {len(chunks)}")
print(f"Persisted at: {PERSIST_DIRECTORY}")

Vector store created successfully.
Stored chunks: 343
Persisted at: c:\Data_science\Modern_route\3) RAG\chroma_llama2_retriever


In [22]:
def display_documents(
    documents,
    max_characters: int = 700
) -> None:
    """
    Display retrieved LangChain Document objects clearly.
    """

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:max_characters])
        print()

In [23]:
# vector_store.similarity_search("What is Llama 2 PAPER all about?")

##### **6.7) Retrieval**

In [24]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

In [25]:
## Regular retrieval using Cosine Similarity
query = "What model sizes of Llama 2 were released?"

similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-12
SOURCE: c:\Data_science\Modern_route\Research\LLAMA2_research_paper.pdf
------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the open release of LLMs, when done safely, will be a net benefit to society. Like all LLMs,
Llama 2 is

RANK: 2
PAPER PAGE: 77
SECTION: appendix
CHUNK ID: llama2-

In [26]:
## Retrieval using Maximal Marginal Relevance (MMR)
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

In [27]:
query = "How was Llama 2-Chat trained and aligned?"

mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents)

RANK: 1
PAPER PAGE: 5
SECTION: pretraining
CHUNK ID: llama2-page-5-chunk-14
SOURCE: c:\Data_science\Modern_route\Research\LLAMA2_research_paper.pdf
------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within distribution.
2 Pretraining
Tocreatethenewfamilyof Llama 2models,webeganwiththepretrainingapproachdes

RANK: 2
PAPER PAGE: 31
SECTION: safety
CHUNK ID: llama2-pag

In [28]:
## Comparing MMR retriever and similarity retriever results for the same query
query = "How was Llama 2-Chat trained and aligned?"

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

similarity_results = similarity_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

In [29]:
print("Similarity Search results:")
for document in similarity_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

print("\nMMR results:")
for document in mmr_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

Similarity Search results:
5 pretraining llama2-page-5-chunk-14
3 introduction llama2-page-3-chunk-9
8 fine_tuning llama2-page-8-chunk-29
4 introduction llama2-page-4-chunk-12

MMR results:
5 pretraining llama2-page-5-chunk-14
31 safety llama2-page-31-chunk-135
77 appendix llama2-page-77-chunk-339
34 discussion llama2-page-34-chunk-146


In [30]:
## pASSING A THRESHOLD TO HAVE A MORE RELEVANT RETRIEVAL
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": 0.64, ## Kind of a measure for accuracy of the retrieval. The higher the threshold, the more relevant the results will be, but it may also reduce the number of results returned.
    }
)

In [31]:
query = "What safety techniques were used for Llama 2-Chat?"

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-11
SOURCE: c:\Data_science\Modern_route\Research\LLAMA2_research_paper.pdf
------------------------------------------------------------------------------------------
Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the
prompt set, subjectivity of the review guidelines, and subjectivity of individual raters. Additionally, these
safety evaluations are performed using content standards that are likely to be biased towards theLlama
2-Chatmodels.
We are releasing the following models to the general publ



In [32]:
## Observe the method being used here. It is similarity_search_with_relevance_scores, which returns a list of tuples containing the document and its relevance score.

query = "What safety techniques were used for Llama 2-Chat?"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)

for rank, (document, relevance_score) in enumerate(
    scored_results,
    start=1
):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Relevance score: {relevance_score:.4f}")
    print(f"Paper page: {document.metadata.get('paper_page')}")
    print(f"Section: {document.metadata.get('section')}")
    print(document.page_content[:500])

Rank: 1
Relevance score: 0.6412
Paper page: 4
Section: introduction
Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the
prompt set, subjectivity of the review guidelines, and subjectivity of individual ra
Rank: 2
Relevance score: 0.6368
Paper page: 3
Section: introduction
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3
Rank: 3
Relevance score: 0.6299
Paper page: 2
Section: front_matter
4 Safety 20
4.1 Safe

In [33]:
metric_query = "How was reinforcement learning with human feedback used?"

candidate_documents = vector_store.similarity_search(
    metric_query,
    k=6,
)

candidate_texts = [
    document.page_content
    for document in candidate_documents
]

print(f"Candidate chunks selected: {len(candidate_texts)}")

Candidate chunks selected: 6


In [34]:
query_vector = np.asarray(
    embeddings.embed_query(metric_query),
    dtype=np.float64,
)

document_vectors = np.asarray(
    embeddings.embed_documents(candidate_texts),
    dtype=np.float64,
)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

Query-vector shape: (1536,)
Document-vectors shape: (6, 1536)


In [35]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b) / denominator
    )


def euclidean_distance(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.linalg.norm(vector_a - vector_b)
    )


def dot_product(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.dot(vector_a, vector_b)
    )

In [36]:
metric_rows = []

for document, document_vector in zip(
    candidate_documents,
    document_vectors
):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "chunk_id": document.metadata.get("chunk_id"),
            "cosine_similarity": cosine_similarity(
                query_vector,
                document_vector
            ),
            "euclidean_distance": euclidean_distance(
                query_vector,
                document_vector
            ),
            "dot_product": dot_product(
                query_vector,
                document_vector
            ),
            "preview": document.page_content[:100].replace(
                "\n",
                " "
            ),
        }
    )

metric_table = pd.DataFrame(metric_rows)

metric_table

,paper_page,section,chunk_id,cosine_similarity,euclidean_distance,dot_product,preview
0,9,fine_tuning,llama2-page-9-chunk-34,0.579209,0.917473,0.579330,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,llama2-page-32-chunk-138,0.538496,0.960757,0.538522,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,llama2-page-17-chunk-73,0.527820,0.971803,0.527842,system message during the conversation by inte...
3,13,fine_tuning,llama2-page-13-chunk-54,0.505250,0.994579,0.505090,evaluating a generative model is an open resea...
4,10,fine_tuning,llama2-page-10-chunk-35,0.500132,0.999793,0.500057,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,llama2-page-10-chunk-40,0.495894,1.004089,0.495886,3.2.2 Reward Modeling The reward model takes a...


In [37]:
metric_query

'How was reinforcement learning with human feedback used?'

In [38]:
metric_table.sort_values(
    by="cosine_similarity",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "cosine_similarity",
        "preview",
    ]
]

,paper_page,section,cosine_similarity,preview
0,9,fine_tuning,0.579209,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.538496,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.527820,system message during the conversation by inte...
3,13,fine_tuning,0.505250,evaluating a generative model is an open resea...
4,10,fine_tuning,0.500132,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,0.495894,3.2.2 Reward Modeling The reward model takes a...


In [39]:
metric_table.sort_values(
    by="euclidean_distance",
    ascending=True
)[
    [
        "paper_page",
        "section",
        "euclidean_distance",
        "preview",
    ]
]

,paper_page,section,euclidean_distance,preview
0,9,fine_tuning,0.917473,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.960757,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.971803,system message during the conversation by inte...
3,13,fine_tuning,0.994579,evaluating a generative model is an open resea...
4,10,fine_tuning,0.999793,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,1.004089,3.2.2 Reward Modeling The reward model takes a...


In [40]:
metric_table.sort_values(
    by="dot_product",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "dot_product",
        "preview",
    ]
]

,paper_page,section,dot_product,preview
0,9,fine_tuning,0.579330,"learning rate of2 × 10−5, a weight decay of 0...."
1,32,discussion,0.538522,"bility, seemed a somewhat shadowy field for th..."
2,17,fine_tuning,0.527842,system message during the conversation by inte...
3,13,fine_tuning,0.505090,evaluating a generative model is an open resea...
4,10,fine_tuning,0.500057,"sampled human preferences, whereby human annot..."
5,10,fine_tuning,0.495886,3.2.2 Reward Modeling The reward model takes a...


In [41]:
normalized_query_vector = (
    query_vector / np.linalg.norm(query_vector)
)

normalized_document_vectors = (
    document_vectors
    / np.linalg.norm(
        document_vectors,
        axis=1,
        keepdims=True
    )
)

normalized_dot_scores = (
    normalized_document_vectors
    @ normalized_query_vector
)

cosine_scores = np.asarray(
    [
        cosine_similarity(
            query_vector,
            document_vector
        )
        for document_vector in document_vectors
    ]
)

print("Cosine scores:")
print(cosine_scores)

print("\nDot product after normalization:")
print(normalized_dot_scores)

print(
    "\nAre they approximately equal?",
    np.allclose(
        cosine_scores,
        normalized_dot_scores,
        atol=1e-8,
    ),
)

Cosine scores:
[0.57920916 0.53849574 0.52781971 0.50524979 0.50013172 0.49589417]

Dot product after normalization:
[0.57920916 0.53849574 0.52781971 0.50524979 0.50013172 0.49589417]

Are they approximately equal? True


##### **6.8) Metadata Filtering**

In [42]:
fine_tuning_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "section": "fine_tuning"
        },
    }
)

In [43]:
query = "How was Llama 2-Chat aligned with human preferences?"

fine_tuning_documents = fine_tuning_retriever.invoke(query)

display_documents(fine_tuning_documents)

RANK: 1
PAPER PAGE: 19
SECTION: fine_tuning
CHUNK ID: llama2-page-19-chunk-79
SOURCE: c:\Data_science\Modern_route\Research\LLAMA2_research_paper.pdf
------------------------------------------------------------------------------------------
Figure12: Humanevaluationresults for Llama 2-Chatmodelscomparedtoopen-andclosed-sourcemodels
across ~4,000 helpfulness prompts with three raters per prompt.
The largestLlama 2-Chat model is competitive with ChatGPT.Llama 2-Chat 70B model has a win rate of
36% and a tie rate of 31.5% relative to ChatGPT.Llama 2-Chat 70B model outperforms PaLM-bison chat
model by a large percentage on our prompt set. More results and analysis is available in Section A.3.7.
Inter-Rater Reliability (IRR). In our human evaluations, three different annotators provided independent
assessments for each model generation comparison. High IRR scores (closer to 1.0) are typically seen as
better from a data quality persp

RANK: 2
PAPER PAGE: 18
SECTION: fine_tuning
CHUNK ID: lla

In [44]:
for document in fine_tuning_documents:
    assert document.metadata["section"] == "fine_tuning"

print("All returned documents are from the fine_tuning section.")

All returned documents are from the fine_tuning section.


In [45]:
## Additional filters added to above i.e. Metadata pre-filtering. 
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {
                    "section": {
                        "$eq": "fine_tuning"
                    }
                },
                {
                    "year": {
                        "$eq": 2023
                    }
                },
                {
                    "organization": {
                        "$eq": "Meta"
                    }
                },
            ]
        },
    }
)

In [46]:
query = "How was human preference data collected?"

filtered_documents = filtered_retriever.invoke(query)

display_documents(filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-35
SOURCE: c:\Data_science\Modern_route\Research\LLAMA2_research_paper.pdf
------------------------------------------------------------------------------------------
sampled human preferences, whereby human annotators select which of two model outputs they prefer.
This human feedback is subsequently used to train a reward model, which learns patterns in the preferences
of the human annotators and can then automate preference decisions.
3.2.1 Human Preference Data Collection
Next, we collect human preference data for reward modeling. We chose a binary comparison protocol over
other schemes, mainly because it enables us to maximize the diversity of collected prompts. Still, other
strategies are worth considering, which we leave for future work.
Our annotation procedure proceeds as follows. We ask annotators to first write a prompt, then choose
between two 

RANK: 2
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: lla

Pre-filtering

In [47]:
pre_filter = {
    "$and": [
        {
            "section": {
                "$eq": "fine_tuning"
            }
        },
        {
            "year": {
                "$eq": 2023
            }
        },
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": pre_filter,
    }
)

pre_filtered_documents = pre_filtered_retriever.invoke(
    "How was the reward model trained?"
)

display_documents(pre_filtered_documents)

RANK: 1
PAPER PAGE: 13
SECTION: fine_tuning
CHUNK ID: llama2-page-13-chunk-54
SOURCE: c:\Data_science\Modern_route\Research\LLAMA2_research_paper.pdf
------------------------------------------------------------------------------------------
evaluating a generative model is an open research question, the ranking task of the reward has no ambiguity.
Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat.
3.2.3 Iterative Fine-Tuning
As we received more batches of human preference data annotation, we were able to train better reward
models and collect more prompts. We therefore trained successive versions for RLHF models, referred to
here as RLHF-V1, ..., RLHF-V5.
We explored RLHF fine-tuning with two main algorithms:
• Proximal Policy Optimization (PPO)(Schulman et al., 2017), the standard in RLHF literature.
• RejectionSamplingfine-tuning . Wesampl

RANK: 2
PAPER PAGE: 11
SECTION: fine_tuning
CHUNK ID: lla

Post-filtering

In [48]:
unfiltered_candidates = vector_store.similarity_search(
    query="How was the reward model trained?",
    k=15,
)

In [49]:
post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "fine_tuning"
    and document.metadata.get("year") == 2023
]

display_documents(post_filtered_documents)

RANK: 1
PAPER PAGE: 13
SECTION: fine_tuning
CHUNK ID: llama2-page-13-chunk-54
SOURCE: c:\Data_science\Modern_route\Research\LLAMA2_research_paper.pdf
------------------------------------------------------------------------------------------
evaluating a generative model is an open research question, the ranking task of the reward has no ambiguity.
Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat.
3.2.3 Iterative Fine-Tuning
As we received more batches of human preference data annotation, we were able to train better reward
models and collect more prompts. We therefore trained successive versions for RLHF models, referred to
here as RLHF-V1, ..., RLHF-V5.
We explored RLHF fine-tuning with two main algorithms:
• Proximal Policy Optimization (PPO)(Schulman et al., 2017), the standard in RLHF literature.
• RejectionSamplingfine-tuning . Wesampl

RANK: 2
PAPER PAGE: 11
SECTION: fine_tuning
CHUNK ID: lla

In [50]:
print(
    "Documents retrieved before post-filtering:",
    len(unfiltered_candidates),
)

print(
    "Documents remaining after post-filtering:",
    len(post_filtered_documents),
)

Documents retrieved before post-filtering: 15
Documents remaining after post-filtering: 13


####

In [51]:
import os
from typing import List

from pydantic import BaseModel, Field

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import (
    CrossEncoderReranker,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [52]:
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 600,
) -> None:
    """Display retrieved LangChain Document objects."""

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(
        documents[:max_documents],
        start=1,
    ):
        metadata = document.metadata

        print(f"\nRANK: {rank}")
        print(f"Paper page: {metadata.get('paper_page')}")
        print(f"Section: {metadata.get('section')}")
        print(f"Chunk ID: {metadata.get('chunk_id')}")
        print("-" * 100)
        print(document.page_content[:max_characters])

In [53]:
def deduplicate_documents(documents):
    """Remove duplicate retrieved chunks while preserving their order."""

    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = (
            document.metadata.get("chunk_id")
            or (
                document.metadata.get("source"),
                document.metadata.get("page"),
                document.page_content,
            )
        )

        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents

In [ ]:
from langchain_community.retrievers import BM25Retriever
# Updated version of TF-IDF.

In [55]:
bm25_retriever = BM25Retriever.from_documents(chunks)

In [56]:
bm25_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000270DFDB3C90>)

In [57]:
# Final number of results
bm25_retriever.k = 4

In [58]:
sparse_query = "Grouped-Query Attention GQA 70B"

In [59]:
sparse_documents = bm25_retriever.invoke(sparse_query)

In [60]:
display_documents(
    sparse_documents,
    title="Sparse Retrieval: BM25 Results",
)


Sparse Retrieval: BM25 Results

RANK: 1
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params Context
Length
GQA Tokens LR
Llama 1 See Touvron et al.
(2023)
7B 2k ✗ 1.0T 3.0 × 10−4
13B 2k ✗ 1.0T 3.0 × 10−4
33B 2k ✗ 1.4T 1.5 × 10−4
65B 2k ✗ 1.4T 1.5 × 10−4
Llama 2 A new mix of publicly
available online data
7B 4k ✗ 2.0T 3.0 × 10−4
13B 4k ✗ 2.0T 3.0 × 10−4
34B 4k ✓ 2.0T 1.5 × 10−4
70B 4k ✓ 2.0T 1.5 × 10−4
Table 1:Llama 2 family of models.Token counts refer to pretraining data only. All models are trained with
a global batch-size of 4M tokens. Bigger models — 34B and 70B — use Grouped-Query Attention (GQA) for
improved inference scalability.
0 250 500 750 1000 1250 15

RANK: 2
Paper page: 48
Section: appendix
Chunk ID: llama2-page-48-chunk-220
----------------------------------------------------------------------------------------------------
BoolQ PIQA 

In [61]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

In [62]:
dense_query = (
    "How did Meta improve inference scalability "
    "for the largest Llama 2 models?"
)

dense_documents = dense_retriever.invoke(dense_query)

display_documents(
    dense_documents,
    title="Dense Retrieval: Vector Search Results",
)


Dense Retrieval: Vector Search Results

RANK: 1
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-15
----------------------------------------------------------------------------------------------------
Specifically, we performed more robust data cleaning, updated our data mixes, trained on 40% more total
tokens,doubledthecontextlength,andusedgrouped-queryattention(GQA)toimproveinferencescalability
for our larger models. Table 1 compares the attributes of the newLlama 2 models with theLlama 1 models.
2.1 Pretraining Data
Our training corpus includes a new mix of data from publicly available sources, which does not include data
from Meta’s products or services. We made an effort to remove data from certain sites known to contain a
high volume of personal information about private individuals. 

RANK: 2
Paper page: 77
Section: appendix
Chunk ID: llama2-page-77-chunk-340
----------------------------------------------------------------------------------------------------
Lla

In [63]:
comparison_query = (
    "How did grouped-query attention improve "
    "Llama 2 inference scalability?"
)

sparse_results = bm25_retriever.invoke(comparison_query)
dense_results = dense_retriever.invoke(comparison_query)

display_documents(
    sparse_results,
    title="BM25 Results",
    max_documents=4,
)

display_documents(
    dense_results,
    title="Dense Vector Results",
    max_documents=4,
)


BM25 Results

RANK: 1
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the

RANK: 2
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-217
----------------------------------------------------------------------------------------------------
attention (MHA) models grow 

In [64]:
print("SPARSE RESULTS")
for rank, document in enumerate(sparse_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

print("\nDENSE RESULTS")
for rank, document in enumerate(dense_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

SPARSE RESULTS
1 4 llama2-page-4-chunk-12
2 47 llama2-page-47-chunk-217
3 54 llama2-page-54-chunk-240
4 6 llama2-page-6-chunk-18

DENSE RESULTS
1 13 llama2-page-13-chunk-53
2 48 llama2-page-48-chunk-221
3 5 llama2-page-5-chunk-15
4 6 llama2-page-6-chunk-18


In [65]:
bm25_retriever.k = 8

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 8
    },
)

In [66]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever,
    ],
    weights=[
        0.5,  # BM25 weight
        0.5,  # Dense-retrieval weight
    ],
)

In [67]:
hybrid_query = (
    "Llama 2 70B grouped-query attention and inference scalability"
)

In [68]:
hybrid_documents = hybrid_retriever.invoke(hybrid_query)

In [69]:
display_documents(
    hybrid_documents,
    title="Hybrid Retrieval: BM25 + Dense + RRF",
    max_documents=6,
)


Hybrid Retrieval: BM25 + Dense + RRF

RANK: 1
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the

RANK: 2
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Trai

In [70]:
test_query = (
    "How did Meta make Llama 2 70B efficient for large-scale inference?"
)

In [71]:
sparse_results = bm25_retriever.invoke(test_query)
dense_results = dense_retriever.invoke(test_query)
hybrid_results = hybrid_retriever.invoke(test_query)

In [72]:
display_documents(
    sparse_results,
    title="1. Sparse Retrieval",
    max_documents=4,
)

display_documents(
    dense_results,
    title="2. Dense Retrieval",
    max_documents=4,
)

display_documents(
    hybrid_results,
    title="3. Hybrid Retrieval",
    max_documents=4,
)


1. Sparse Retrieval

RANK: 1
Paper page: 54
Section: appendix
Chunk ID: llama2-page-54-chunk-240
----------------------------------------------------------------------------------------------------
attribute, and so, up to 20 turns (we did not extend the human evaluation more, and all the examples had
less than 4048 tokens in total over the turns). As a comparison,Llama 2-Chat without GAtt can not anymore
refer to the attributes after only few turns: from 100% at turn t+1, to 10% at turn t+3 and then 0%.
GAtt Zero-shot Generalisation. We tried at inference time to set constrain not present in the training of
GAtt. For instance, “answer in one sentence only”, for which the model remained consistent, as illustrated in
Figure 28.
We applied first GAtt toLlama 1, which was pretrained with a 

RANK: 2
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params C

In [73]:
CHAT_MODEL = os.environ.get(
    "OPENAI_CHAT_MODEL",
    "gpt-4.1-mini",
)

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
)

print("Chat model:", CHAT_MODEL)

Chat model: gpt-4.1-mini


In [74]:
query_rewriting_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You rewrite conversational questions into clear,
standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Return only one rewritten query.
""",
        ),
        (
            "human",
            """
Conversation history:
{chat_history}

Current query:
{query}
""",
        ),
    ]
)

In [75]:
query_rewriting_chain = (query_rewriting_prompt| llm | StrOutputParser())

In [76]:
chat_history = """
User: How was Llama 2-Chat initially fine-tuned?
Assistant: It first underwent supervised fine-tuning.
"""

In [77]:
original_query = "What did Meta do after that?"

In [78]:
rewritten_query = query_rewriting_chain.invoke(
    {
        "chat_history": chat_history,
        "query": original_query,
    }
).strip()

In [79]:
print("Original query:")
print(original_query)

print("\nRewritten query:")
print(rewritten_query)

Original query:
What did Meta do after that?

Rewritten query:
What steps did Meta take after the initial supervised fine-tuning of Llama 2-Chat?


In [80]:
rewritten_query_documents = hybrid_retriever.invoke(
    rewritten_query
)

In [81]:
display_documents(
    rewritten_query_documents,
    title="Documents Retrieved Using the Rewritten Query",
    max_documents=5,
)


Documents Retrieved Using the Rewritten Query

RANK: 1
Paper page: 8
Section: fine_tuning
Chunk ID: llama2-page-8-chunk-29
----------------------------------------------------------------------------------------------------
are from OpenAI (2023). Results for the PaLM model are from Chowdhery et al. (2022). Results for the
PaLM-2-L are from Anil et al. (2023).
3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and RLHF, requiring significant computational and annotation resources.
In this section, we report on our experiments and findings using supervised fine-tuning (Section 3.1), as
well as initial and iterative reward modeling (Section 3.2.2) and RLHF (Section 3.2.3). We also share a
new technique, Ghost A

RANK: 2
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
-------------------------------------------------------------------------------------------------

In [82]:
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description=(
            "Four alternative search queries expressing "
            "the same information need using different wording."
        )
    )

In [83]:
query_expansion_llm = llm.with_structured_output(ExpandedQueryOutput)

In [84]:
query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.
""",
        ),
        (
            "human",
            "Original query: {query}",
        ),
    ]
)

In [85]:
query_expansion_chain = (query_expansion_prompt| query_expansion_llm)

In [86]:
original_query = (
    "How was Llama 2-Chat improved using human feedback?"
)
